In [1]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('NbAiLab/nb-sbert-base', device="cuda")

# Enkel vs dobbel input

In [2]:
enc_1 = model.encode(["hei", ""])
enc_2 = model.encode(["hei", "hei"])

enc_1.shape, enc_2.shape

((2, 768), (2, 768))

In [3]:
import numpy as np 

np.all(enc_1[0] == enc_2[1])

True

In [4]:
np.all(enc_1[0] == enc_2[0])

True

In [5]:
enc_1 = model.encode(["hei"])
enc_2 = model.encode(["hei", "hei"])

enc_1.shape, enc_2.shape

((1, 768), (2, 768))

In [6]:
np.all(enc_1[0] == enc_2[1])

False

In [7]:
np.all(enc_1[0] == enc_2[0])

False

Men de er tilnærmet like, da

In [8]:
enc_1[0][:50], enc_2[0][:50]

(array([ 0.3647799 ,  0.34224588, -0.34396833,  0.00879236,  0.32044986,
         0.54916227, -0.10220651, -0.45618382,  0.12647659,  0.49583614,
         1.0172874 , -0.11790139,  0.4303897 ,  0.13077766,  0.0985949 ,
         0.05770493,  0.28913295,  1.143943  ,  0.7058268 ,  0.21631092,
         0.4849031 , -0.21270666, -1.3980572 ,  0.38342652, -0.4407696 ,
        -0.03321898, -0.10697224, -0.43914795,  1.5083432 , -0.5273925 ,
         0.13471957,  0.31760615, -0.624584  ,  0.65706253, -0.17938213,
         0.4257838 , -0.65206003, -0.14552468, -0.6551271 ,  0.65032375,
        -0.40146327,  0.8722882 , -0.702847  ,  0.88905156, -0.12251735,
         0.5888145 ,  0.06136187,  0.7564927 , -1.7971756 , -0.6471126 ],
       dtype=float32),
 array([ 0.36477983,  0.342246  , -0.3439683 ,  0.00879224,  0.32044986,
         0.54916227, -0.10220663, -0.45618406,  0.12647656,  0.49583662,
         1.0172876 , -0.11790141,  0.43038923,  0.13077784,  0.09859565,
         0.05770482,  0.289

# Uviss aggregering/encoding hack
Dokumentasjonen til sentence_transformers godtar en streng, eller en liste med strenger som input til model.encode  
Men, jeg fant ut at man kan sende inn en liste av liste med strenger, og at denne representasjonen er bedre enn å sende hver liste inn separat og så aggregere opp!!!  
Dette må undersøkes videre

In [9]:
model.encode(["foo", "bar", "baz"]).shape, model.encode([["foo", "bar", "baz"]]).shape

((3, 768), (1, 768))

In [10]:
s = "Marie Joys var en norsk sykepleier. Hun var en pioner innenfor norsk sykepleierutdanning."
s_delt = ["Marie Joys var en norsk sykepleier.", "Hun var en pioner innenfor norsk sykepleierutdanning."]

Vanlig setning

In [12]:
enc_vanlig = model.encode(s)
enc_vanlig.shape

(768,)

Delt setning

In [13]:
enc_delt = model.encode(s_delt)
enc_delt.shape

(2, 768)

Encoding hack 

In [14]:
enc_hack = model.encode([s_delt])
enc_hack.shape

(1, 768)

Aggreger delt setning med mean pooling

In [15]:
mp_enc_delt = np.mean(enc_delt, axis=0)
mp_enc_delt.shape

(768,)

Sammenlikn vektorliket mellom original og aggregerte setninger:

In [16]:
util.cos_sim(enc_vanlig, mp_enc_delt)

tensor([[0.9637]])

In [17]:
util.cos_sim(enc_vanlig, enc_hack[0])

tensor([[0.9861]])

Den "hacky" varianten er likere enn mean pooling-varianten

## Hvorfor er det sånn?
La oss se i kildekoden  
Hentet fra https://github.com/UKPLab/sentence-transformers/blob/master/sentence_transformers/SentenceTransformer.py  
Dato: 2024-02-21  
Bare lagt til litt print her og der

In [18]:
from typing import Union, List
import torch 
from torch import Tensor, device 
from numpy import ndarray
from tqdm.autonotebook import trange
import logging

logger = logging.getLogger(__name__)

def batch_to_device(batch, target_device: device):
    """
    send a pytorch batch to a device (CPU/GPU)
    """
    for key in batch:
        if isinstance(batch[key], Tensor):
            batch[key] = batch[key].to(target_device)
    return batch


def encode(
    model,
    sentences: Union[str, List[str]],
    batch_size: int = 32,
    show_progress_bar: bool = None,
    output_value: str = "sentence_embedding",
    convert_to_numpy: bool = True,
    convert_to_tensor: bool = False,
    device: str = None,
    normalize_embeddings: bool = False,
) -> Union[List[Tensor], ndarray, Tensor]:
    """
    Computes sentence embeddings.

    :param sentences: the sentences to embed.
    :param batch_size: the batch size used for the computation.
    :param show_progress_bar: Whether to output a progress bar when encode sentences.
    :param output_value: The type of embeddings to return: "sentence_embedding" to get sentence embeddings,
        "token_embeddings" to get wordpiece token embeddings, and `None`, to get all output values. Defaults
        to "sentence_embedding".
    :param convert_to_numpy: Whether the output should be a list of numpy vectors. If False, it is a list of PyTorch tensors.
    :param convert_to_tensor: Whether the output should be one large tensor. Overwrites `convert_to_numpy`.
    :param device: Which `torch.device` to use for the computation.
    :param normalize_embeddings: Whether to normalize returned vectors to have length 1. In that case,
        the faster dot-product (util.dot_score) instead of cosine similarity can be used.

    :return: By default, a list of tensors is returned. If convert_to_tensor, a stacked tensor is returned.
        If convert_to_numpy, a numpy matrix is returned.
    """
    model.eval()
    if show_progress_bar is None:
        show_progress_bar = (
            logger.getEffectiveLevel() == logging.INFO or logger.getEffectiveLevel() == logging.DEBUG
        )

    if convert_to_tensor:
        convert_to_numpy = False

    if output_value != "sentence_embedding":
        convert_to_tensor = False
        convert_to_numpy = False

    input_was_string = False
    if isinstance(sentences, str) or not hasattr(
        sentences, "__len__"
    ):  # Cast an individual sentence to a list with length 1
        sentences = [sentences]
        input_was_string = True

    if device is None:
        device = model.device

    print("normalize embeddings:", normalize_embeddings)
    print("convert_to_numpy:", convert_to_numpy)
    print("convert_to_tensor:", convert_to_tensor)
    print(f"input_was_string: {input_was_string}")
    

    model.to(device)

    all_embeddings = []
    length_sorted_idx = np.argsort([-model._text_length(sen) for sen in sentences])
    sentences_sorted = [sentences[idx] for idx in length_sorted_idx]

    print(f"senteces_sorted: {sentences_sorted}")
    print(f"len(sentences): {len(sentences)}")

    for start_index in trange(0, len(sentences), batch_size, desc="Batches", disable=not show_progress_bar):
        sentences_batch = sentences_sorted[start_index : start_index + batch_size]
        features = model.tokenize(sentences_batch)
        print(f"features: {features}")
        features = batch_to_device(features, device)
        # print(f"features: {features}")

        with torch.no_grad():
            out_features = model.forward(features)

            if output_value == "token_embeddings":
                embeddings = []
                for token_emb, attention in zip(out_features[output_value], out_features["attention_mask"]):
                    last_mask_id = len(attention) - 1
                    while last_mask_id > 0 and attention[last_mask_id].item() == 0:
                        last_mask_id -= 1

                    embeddings.append(token_emb[0 : last_mask_id + 1])
            elif output_value is None:  # Return all outputs
                embeddings = []
                for sent_idx in range(len(out_features["sentence_embedding"])):
                    row = {name: out_features[name][sent_idx] for name in out_features}
                    embeddings.append(row)
            else:  # Sentence embeddings
                embeddings = out_features[output_value]
                embeddings = embeddings.detach()
                if normalize_embeddings:
                    embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

                # fixes for #522 and #487 to avoid oom problems on gpu with large datasets
                if convert_to_numpy:
                    embeddings = embeddings.cpu()

            all_embeddings.extend(embeddings)

    all_embeddings = [all_embeddings[idx] for idx in np.argsort(length_sorted_idx)]

    if convert_to_tensor:
        if len(all_embeddings):
            all_embeddings = torch.stack(all_embeddings)
        else:
            all_embeddings = torch.Tensor()
    elif convert_to_numpy:
        all_embeddings = np.asarray([emb.numpy() for emb in all_embeddings])

    if input_was_string:
        all_embeddings = all_embeddings[0]

    return all_embeddings

### Se på forskjellene i encode med forskjellig input

In [19]:
enc_delt = encode(model, s_delt)
enc_delt.shape

normalize embeddings: False
convert_to_numpy: True
convert_to_tensor: False
input_was_string: False
senteces_sorted: ['Hun var en pioner innenfor norsk sykepleierutdanning.', 'Marie Joys var en norsk sykepleier.']
len(sentences): 2
features: {'input_ids': tensor([[  101, 16250, 10299, 10110, 24109, 38219, 57875, 16034, 12261, 10550,
         22238, 11709, 11159, 12146, 11269,   119,   102],
        [  101, 11834, 32718, 10107, 10299, 10110, 16034, 12261, 10550, 22238,
         11709,   119,   102,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]])}


(2, 768)

In [20]:
enc_hack = encode(model, [s_delt])
enc_hack.shape

normalize embeddings: False
convert_to_numpy: True
convert_to_tensor: False
input_was_string: False
senteces_sorted: [['Marie Joys var en norsk sykepleier.', 'Hun var en pioner innenfor norsk sykepleierutdanning.']]
len(sentences): 1
features: {'input_ids': tensor([[  101, 11834, 32718, 10107, 10299, 10110, 16034, 12261, 10550, 22238,
         11709,   119,   102, 16250, 10299, 10110, 24109, 38219, 57875, 16034,
         12261, 10550, 22238, 11709, 11159, 12146, 11269,   119,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1]])}


(1, 768)

In [21]:
enc_hel = encode(model, [s])
enc_hel.shape

normalize embeddings: False
convert_to_numpy: True
convert_to_tensor: False
input_was_string: False
senteces_sorted: ['Marie Joys var en norsk sykepleier. Hun var en pioner innenfor norsk sykepleierutdanning.']
len(sentences): 1
features: {'input_ids': tensor([[  101, 11834, 32718, 10107, 10299, 10110, 16034, 12261, 10550, 22238,
         11709,   119, 16250, 10299, 10110, 24109, 38219, 57875, 16034, 12261,
         10550, 22238, 11709, 11159, 12146, 11269,   119,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1]])}


(1, 768)

In [22]:
                                                                                                                               # vvv denne '102' er eneste diff i input_ids tensor (SEP token)
enc_hack_feats =  {'input_ids': torch.tensor([[  101, 11834, 32718, 10107, 10299, 10110, 16034, 12261, 10550, 22238, 11709, 119, 102, 16250, 
                                               10299, 10110, 24109, 38219, 57875, 16034, 12261, 10550, 22238, 11709, 11159, 12146, 11269, 119, 102]]), 
         'token_type_ids': torch.tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 
         'attention_mask': torch.tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

enc_hel_feats = {'input_ids': torch.tensor([[  101, 11834, 32718, 10107, 10299, 10110, 16034, 12261, 10550, 22238, 11709, 119, 16250, 
                                             10299, 10110, 24109, 38219, 57875, 16034, 12261, 10550, 22238, 11709, 11159, 12146, 11269, 119, 102]]), 
         'token_type_ids': torch.tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 
         'attention_mask': torch.tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


enc_hack_feats['input_ids'].shape, enc_hel_feats['input_ids'].shape

(torch.Size([1, 29]), torch.Size([1, 28]))

In [23]:
model.tokenizer.decode(enc_hack_feats['input_ids'][0])

'[CLS] Marie Joys var en norsk sykepleier. [SEP] Hun var en pioner innenfor norsk sykepleierutdanning. [SEP]'

In [24]:
model.tokenizer.decode(enc_hel_feats['input_ids'][0])

'[CLS] Marie Joys var en norsk sykepleier. Hun var en pioner innenfor norsk sykepleierutdanning. [SEP]'

In [25]:
enc_hack_feats["token_type_ids"]

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1]])

Når man sender inn en liste av liste av strenger vil det legges in et SEP-token mellom listene og token_type_ids vil mappe 0-er til tokensa i første og 1-ere til tokensa i andre liste

### Model.tokenize fjerner alt unntatt de to første bitene av tekstene 
Probably fordi det er kun 2 typer token type ids  (https://huggingface.co/docs/transformers/glossary#token-type-ids)

Se metoden tokenize i https://github.com/UKPLab/sentence-transformers/blob/master/sentence_transformers/models/Transformer.py#L131  

Sentence_transformers.encode godtar kun input `str` eller `list[str]`, men model.tokenize godtar også `list[tuple[str, str]]`

In [26]:
model.tokenize([["Marie Joys var en norsk sykepleier", "og sykepleielærer.", "Hun var en pioner innenfor norsk sykepleierutdanning."]])

{'input_ids': tensor([[  101, 11834, 32718, 10107, 10299, 10110, 16034, 12261, 10550, 22238,
          11709,   102, 10156, 12261, 10550, 22238, 19428, 49466,   119,   102]]),
 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1]]),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [27]:
model.tokenizer.decode(model.tokenize([["Marie Joys var en norsk sykepleier", "og sykepleielærer.", "Hun var en pioner innenfor norsk sykepleierutdanning."]])["input_ids"][0])

'[CLS] Marie Joys var en norsk sykepleier [SEP] og sykepleielærer. [SEP]'

### Får vi lenger kontekstlengde med hack-versjonen?

In [28]:
s1 = "Marie Joys var en norsk sykepleier og sykepleielærer. Hun var en pioner innenfor norsk sykepleierutdanning. Hun var oversykepleier ved Bergen kommunale sykehus i 35 år og var blant stifterne av Norsk Sykepleierskeforbund (dagens Norsk Sykepleierforbund) i 1912"
s1_v2 = "Marie Joys var en norsk sykepleier og sykepleielærer. Hun var en pioner innenfor norsk sykepleierutdanning. Hun var oversykepleier ved Bergen kommunale sykehus i 35 år og var blant stifterne av Norsk Sykepleierskeforbund (dagens Norsk Sykepleierforbund) i 1912."


len(model.tokenizer.tokenize(s1)), len(model.tokenizer.tokenize(s1_v2))

(75, 76)

#### "Vanlig" tokenisering
Antall tokens kuttes av ved modellens makslengde

In [29]:
model.tokenize([s1])["input_ids"].shape, model.tokenize([s1_v2])["input_ids"].shape,

(torch.Size([1, 75]), torch.Size([1, 75]))

#### "Hacky" tokenisering
Sender med to tekster som er lik makslengde

In [30]:
s2 = "Marie Joys vokste opp i Bergen. Etter avsluttet folkeskole ble hun av sin far rådet til å utdanne seg i utlandet, slik at hun i tillegg kunne lære et fremmed språk. Noe tilfeldig ble den unge Marie oppmerksom på sykepleierskolen Victoriahaus i Berlin. Skolen var opprettet av keiserinne Viktoria"

len(model.tokenizer.tokenize(s2)) 

75

In [31]:
model.tokenize([[s1, s2]])

{'input_ids': tensor([[   101,  11834,  32718,  10107,  10299,  10110,  16034,  12261,  10550,
           22238,  11709,  10156,  12261,  10550,  22238,  19428,  49466,    119,
           16250,  10299,  10110,  24109,  38219,  57875,  16034,  12261,  10550,
           22238,  11709,  11159,  12146,  11269,    119,  16250,  10299,  10491,
           16105,    102,  11834,  32718,  10107, 105215,  15153,    177,  19511,
             119,  17966,  10170,  64329,  18694,  10171,  44377,  10718,  12041,
           10170,  10795,  13301,  88407,  10308,  10275,    259,  11735,  12146,
           10238,  12042,    177,  11735,  30517,    117,  22962,  10160,  12041,
             177,  26532,    102]]),
 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1]]),
 'attent

In [32]:
model.tokenize([[s1, s2]])["input_ids"].shape

torch.Size([1, 75])

#### Svar: Nei, men hver av de to input-tekstene konkateneres på halvparten av inputlengden

In [33]:
model.tokenizer.decode(model.tokenize([[s1, s2]])["input_ids"][0])

'[CLS] Marie Joys var en norsk sykepleier og sykepleielærer. Hun var en pioner innenfor norsk sykepleierutdanning. Hun var oversy [SEP] Marie Joys vokste opp i Bergen. Etter avsluttet folkeskole ble hun av sin far rådet til å utdanne seg i utlandet, slik at hun i tillegg [SEP]'